# 스포츠로만 배운 모델에 날씨를 넣으면? — 톤이 따라올까
내 모델은 **스포츠 리포트만** 보고 MBC 앵커 리드 말투를 배웠어요. 한 번도 본 적 없는 **날씨**를 넣어도 그 말투가 나올까요?

| 부분 | 넣는 것 | 정답 | 보는 점 |
|---|---|---|---|
| ① 날씨 리포트 | MBC 뉴스데스크 날씨 리포트 5건 (`test_weather.jsonl`) | 실제 앵커 리드 | 주제가 바뀌어도 리드 형식·말투가 유지되나 |
| ② 날씨 데이터 | Open-Meteo 실시간 예보 숫자 | 없음 | 숫자만 주고 기사를 쓰게 하면? 숫자를 틀리지 않나 |

준비: 4번 노트북과 같아요 (Ollama에 `qwen2.5:3b`·내 모델, `.env`의 `OPENAI_API_KEY`). `test_weather.jsonl`을 이 폴더에 넣으세요.

In [8]:
# ▶ 1. 환경 준비 · 공통 함수 — 여기만 고치세요
import os, re, json, html, requests
from datetime import datetime
from pathlib import Path
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

MY_MODEL = 'mbcsports'      # ollama list 에 나온 내 모델 이름
RERUN    = False            # True: 저장된 답을 무시하고 다시 묻기

load_dotenv()
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
cloud = OpenAI(api_key=os.getenv('OPENAI_API_KEY')) if os.getenv('OPENAI_API_KEY') else None
MODELS = {'원본 Qwen2.5-3B': (local, 'qwen2.5:3b'),
          '내가 학습한 모델': (local, MY_MODEL),
          'gpt-4.1-mini':    (cloud, 'gpt-4.1-mini')}
COLORS = {'원본 Qwen2.5-3B': '#9aa5b1', '내가 학습한 모델': '#1f8a70', 'gpt-4.1-mini': '#2563a8'}
pd.set_option('display.max_colwidth', None)

def ask(client, model, instruction, text):
    try:
        r = client.chat.completions.create(model=model, temperature=0,
                messages=[{'role': 'user', 'content': instruction + '\n\n' + text}])
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'(실패: {type(e).__name__})'

def ask_all(df, cache):
    cache = Path(cache)
    if cache.exists() and not RERUN:
        print('저장된 답을 불러왔어요:', cache, '(다시 물으려면 RERUN = True)')
        return json.loads(cache.read_text(encoding='utf-8'))
    out = {}
    for name, (c, m) in MODELS.items():
        print('묻는 중:', name)
        out[name] = [ask(c, m, r['instruction'], r['input']) if c else '(키 없음)' for _, r in df.iterrows()]
    cache.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding='utf-8')
    print('✅ 저장:', cache)
    return out

def failed(a): return (not a) or a.startswith('(실패') or a.startswith('(키')

def checks(ans, src, gold=None):
    a = ans.replace('\\n', '\n')
    sentences = [s for s in re.split(r'(?<=[.!?다요])\s+', a.strip()) if s.strip()]
    extra = sorted(n for n in set(re.findall(r'\d[\d,.]*', a)) if n.strip('.,') not in src)
    greet = bool(re.search(r'MBC\s*뉴스\s*\S{2,4}입니다|안녕하세요', a))
    c = {'끝인사 없음': not greet, '2문장 이하': len(sentences) <= 2, '본문에 없는 숫자 없음': not extra,
         '_문장수': len(sentences), '_숫자': extra, '_인사': greet, '_글자': len(a)}
    if gold:
        c['_배율'] = len(a) / max(len(gold), 1)
        c['길이 비슷(0.5~2배)'] = 0.5 <= c['_배율'] <= 2
    return c

def summary(df, answers, gold_col='output'):
    rows = []
    for name, outs in answers.items():
        res = [checks(a, r['input'], r.get(gold_col)) for a, (_, r) in zip(outs, df.iterrows()) if not failed(a)]
        n = len(res); row = {'모델': name, '답한 문제': f'{n}/{len(outs)}'}
        for k in ['끝인사 없음', '2문장 이하', '길이 비슷(0.5~2배)', '본문에 없는 숫자 없음']:
            if res and k in res[0]: row[k] = f'{sum(x[k] for x in res)}/{n}'
        row['평균 글자 수'] = round(sum(x['_글자'] for x in res) / max(n, 1))
        rows.append(row)
    return pd.DataFrame(rows).set_index('모델')

def badges(c):
    out = [('기자 끝인사·인사말 있음' if c['_인사'] else '끝인사 없음', c['끝인사 없음']),
           (f"{c['_문장수']}문장", c['2문장 이하'])]
    if '_배율' in c:
        out.append(('길이 비슷' if c['길이 비슷(0.5~2배)'] else f"길이 {c['_배율']:.1f}배", c['길이 비슷(0.5~2배)']))
    out.append(('입력에 없는 숫자 없음' if not c['_숫자'] else '입력에 없는 숫자: ' + ', '.join(c['_숫자'][:3]), c['본문에 없는 숫자 없음']))
    return ''.join(f'<span class="b {"ok" if ok else "no"}">{html.escape(t)}</span>' for t, ok in out)

def make_report(df, answers, path, title, note, gold_col='output', src_label='리포트 본문'):
    blocks = []
    for i, (_, r) in enumerate(df.iterrows()):
        gold = r.get(gold_col) if gold_col in df.columns else None
        cards = ''
        for n in answers:
            a = answers[n][i]
            body = (f'<div class="ans empty">{html.escape(a or "답 없음")}</div>' if failed(a) else
                    f'<div class="ans">{html.escape(a.replace(chr(92)+"n", chr(10)))}</div><div class="meta">{badges(checks(a, r["input"], gold))}</div>')
            cards += (f'<div class="card"><div class="h" style="background:{COLORS.get(n, "#18232f")}">{html.escape(n)}</div>{body}'
                      '<label class="human"><input type="checkbox"> 사실 오류 있음</label><label class="human"><input type="checkbox"> MBC 리드 말투</label></div>')
        head = html.escape(r.get('title', f'문제 {i+1}'))
        gold_html = f'<div class="gold"><b>실제 앵커 리드</b> {html.escape(gold)}</div>' if gold else ''
        blocks.append(f'<h2>{i+1}. {head}</h2><details><summary>{src_label} 보기 ({len(r["input"])}자)</summary>'
                      f'<pre class="q">{html.escape(r["input"])}</pre></details>{gold_html}<div class="row">{cards}</div>')
    sum_html = summary(df, answers, gold_col).reset_index().to_html(index=False, classes='sum', border=0)
    page = f'''<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{html.escape(title)}</title><style>
body{{font-family:"Malgun Gothic",sans-serif;margin:24px auto;max-width:1500px;background:#f7f7f5;color:#222;padding:0 16px}}
h1{{font-size:26px}} h2{{font-size:20px;margin-top:40px;border-top:2px solid #ddd;padding-top:20px}}
.sum{{border-collapse:collapse;background:#fff;margin:10px 0}} .sum th,.sum td{{border:1px solid #ddd;padding:6px 12px;text-align:center}} .sum th{{background:#18232f;color:#fff}}
.q{{background:#fff;border:1px solid #ddd;padding:12px;white-space:pre-wrap;border-radius:8px;line-height:1.6}}
.gold{{background:#fff7ed;border-left:6px solid #e8590c;padding:10px 14px;margin:10px 0;line-height:1.7}}
.row{{display:flex;gap:12px;align-items:stretch}} .card{{flex:1;background:#fff;border:1px solid #ddd;border-radius:10px;overflow:hidden;display:flex;flex-direction:column}}
.h{{color:#fff;font-weight:bold;padding:8px 12px;font-size:17px}} .ans{{white-space:pre-wrap;line-height:1.7;padding:12px;flex:1}} .empty{{color:#aaa;text-align:center}}
.meta{{padding:0 12px 8px;font-size:12px}} .b{{display:inline-block;padding:2px 8px;border-radius:10px;margin:2px 4px 2px 0}}
.ok{{background:#d8f3e8;color:#16664c}} .no{{background:#f6dede;color:#8a2a2a}}
.human{{display:block;font-size:13px;color:#555;padding:4px 12px;border-top:1px dashed #eee}} details summary{{cursor:pointer;color:#2563a8;margin:6px 0}}
</style></head><body><h1>{html.escape(title)}</h1>
<p>만든 시각: {datetime.now():%Y-%m-%d %H:%M} · 원본(회색) → 내가 학습한 모델(초록) → gpt-4.1-mini(파랑) · 초록 배지 = 리드 형식을 지킴</p>
<p>{note}</p><h2 style="border:none;margin-top:16px">자동 체크 요약</h2>{sum_html}{''.join(blocks)}</body></html>'''
    Path(path).write_text(page, encoding='utf-8')
    print('✅ 비교표 완성:', Path(path).resolve())
    if os.name == 'nt':
        os.startfile(Path(path).resolve())
print('준비 완료')

준비 완료


## ① MBC 날씨 리포트 5건 → 리드
학습 때와 같은 형식이에요. 지시문만 "스포츠" → "날씨"로 바꿨어요.

In [ ]:
# ▶ ①-1. 날씨 리포트 불러와서 세 모델에 묻기 (약 1분)
weather = pd.read_json('test_weather.jsonl', lines=True)
# weather['instruction'] = '날씨 리포트 본문을 읽고, 앵커가 읽을 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'
weather['instruction'] = '날씨 리포트 본문을 읽고, MBC 뉴스 기자 리포트 기사를 700자 이상으로 쓰세요.'
ans_w = ask_all(weather, 'answers_weather.json')
pd.DataFrame({'실제 앵커 리드': weather['output'], **ans_w}, index=weather['title'])

저장된 답을 불러왔어요: answers_weather.json (다시 물으려면 RERUN = True)


,실제 앵커 리드,원본 Qwen2.5-3B,내가 학습한 모델,gpt-4.1-mini
title,,,,
겨울 문턱 쌀쌀‥서울 벌써 '첫눈',NaN,"MBC 뉴스의 이지혜 앵커입니다. 오늘의 날씨는 잔뜩 흐린 하늘에서 '진눈깨비'가 휘날립니다. 강원도 정선에서는 오늘 낮 굵은 눈발이 쏟아졌고, 한계령 등 인근 산간 지역 역시 오후 내내 눈이 내렸습니다. 서울에서는 올해 첫눈이 공식 관측되며, 지난해보다 26일 빠르고, 평년 기준보다도 12일이나 이른 기록입니다. \n\n이처럼 빗방울이 떨어지면서 서울 한낮 기온이 7도에 머무르며, 어제보다 더 쌀쌀한 날씨가 이어졌습니다. 만추의 풍경을 즐기려 산을 찾은 등산객들은 매서운 바람에 오들오들 떨어야 했습니다. \n\n특히 북쪽에서 밀려 내려온 기압골이 비와 함께 찬 공기를 동반하면서 기온이 어제보다 2~3도 떨어졌습니다. 경기 북부와 강원도 일부 지역에서는 흐리고 바람도 강하게 불어 체감온도가 영하권에 머물렀습니다. \n\n밤에는 기온이 더 떨어져 강원 내륙과 산지 등 일부 지역에서는 진눈깨비가 더 내릴 것으로 예상됩니다. 비는 내일까지 이어질 전망입니다. \n\n이런 날씨로 인해, 주 중반부터는 평년 기온을 회복할 것으로 기상청은 내다봤습니다. \n\nMBC 뉴스 이지혜 앵커입니다.","강원도와 경기 북부가 오늘 낮부터 눈이 내리면서, 서울까지 비가 내려 쌀쌀한 날씨가 이어지고 있습니다. 기온은 평년보다 2~3도 떨어져, 일부 지역에서는 영하권까지 기온이 떨어졌습니다. 박진준 기자입니다.","오늘 전국에 찬 기운이 한층 더 짙어졌습니다. 강원 산간 지역에는 굵은 눈발이 쏟아졌고, 서울에도 올가을 들어 첫눈이 공식 관측됐습니다. 평년보다 열흘 이상 빠른 기록인데요, 수도권을 중심으로 비와 함께 기온이 크게 떨어지면서 쌀쌀한 날씨가 이어지고 있습니다. 자세한 날씨 소식, 기상캐스터 연결해 전해드립니다."
'입동 추위' 서울 첫 얼음 관측‥내일은 비 소식,NaN,"입동이라는 말의 이름값을 하듯, 초겨울의 추위가 찾아왔습니다. 북서쪽에서 강한 바람과 함께 찬 공기가 서울을 강타했습니다. 서울 종로구 기상관측소에서 포착된 얼음은 올가을 첫 얼음으로, 지난해보다 열흘, 평년보다 나흘 늦게 나타났습니다. 서울의 오전 최저 기온은 0.4도로 올가을 들어 가장 추웠으며, 바람까지 강하게 불어 체감 기온은 영하 2도까지 내려갔습니다. 전국 곳곳이 0도 아래로 뚝 떨어졌습니다. 한낮에도 기온은 9.1도에서 15.9도에 머무르며, 전국이 10도 내외의 큰 일교차를 보였습니다. 강원도 속초 해변에서는 따뜻한 햇볕 덕에 차가운 바닷물에 발을 담그는 모습을 볼 수 있었고, 곳곳에서 여유롭게 해변을 걸을 수 있었습니다. \n\n이제 내일은 중부지방에 비가 내릴 것으로 예상됩니다. 예상 강수량은 5mm 안팎으로 적겠지만, 기온이 낮은 지역에선 첫눈이 내릴 수도 있습니다. 다음 주 수요일부터 기온은 평년 수준을 되찾을 것으로 기상청은 내다봤습니다. \n\nMBC 뉴스 박소희입니다.","입동이라는 말의 이름값을 하듯, 초겨울 추위가 찾아왔습니다. 서울의 올가을 첫 얼음이 지난해보다 10일, 평년보다 4일 늦게 나타났는데요. 기온이 낮은 지역에선 첫눈이 내릴 수도 있습니다. 박소희 기자입니다.","입동을 맞아 초겨울 추위가 본격적으로 찾아왔습니다. 북서쪽에서 찬 공기가 강한 바람과 함께 내려오면서 서울을 비롯한 전국 곳곳에 올가을 들어 가장 낮은 기온이 기록됐는데요. 자세한 날씨 소식, 박소희 기자가 전해드립니다."
'입동 추위' 온다‥내일 서울 첫 영하권 '뚝',NaN,"네, 오늘의 날씨는 정말 겨울이 찾아왔습니다. 광화문에서의 기온은 이미 5도까지 떨어졌고, 강한 바람에 체감온도는 2도 선에 머물고 있습니다. 갑작스러운 추위로 인해 퇴근길 시민들의 옷차림도 한결 두꺼워졌습니다. 이처럼 추위가 찾아온 이유는 중국 북부의 고기압이 한반도로 내려와 찬 공기를 몰고 왔기 때문입니다. 서울 상공의 기온은 하루 새 10도가량 떨어져 영하 5도까지 내려갔습니다. 이에 더해 밤사이 이불 역할을 해주는 구름이 없어, 내일 아침은 더욱 추워질 것으로 예상됩니다. 서울은 올 가을 첫 영하권인 영하 1도까지 내려가며, 전국 내륙 곳곳에는 밤 9시를 기해 한파주의보까지 발효될 것으로 보입니다. 이 추위는 다음 주 초까지 이어질 것으로 예상되며, 주 중반부터는 기온이 다시 평년 수준을 웃돌기 시작할 예정입니다. 이에 건강관리에 신경 쓰시기 바랍니다.","이번 추위가 작년과 재작년보다 빨리 시작되는 것으로 보입니다. 내일 서울이 영하권에 들어가는 등, 기온 변동 폭이 큰 만큼 건강관리에 신경 쓰시기 바랍니다.","네, 오늘 아침부터 갑자기 찾아온 강추위에 시민들 옷차림도 확 달라졌습니다. 서울 광화문 기온이 5도까지 뚝 떨어지고, 강한 바람까지 불면서 체감온도는 2도에 머무르고 있는데요. 올가을 들어 가장 쌀쌀한 날씨, 자세한 소식 기상센터 연결해 전해드립니다."
6년 만의 '11월 황사'‥주말엔 서울 첫 영하권,NaN,"MBC 뉴스 윤태구입니다. 도심 빌딩 너머가 뿌옇게 흐려 보입니다. 이틀 전 내몽골 고원에서 발원한 황사가 바람을 타고 날아들었습니다. 수도권의 미세먼지 농도는 한때 '매우나쁨' 수준까지 치솟았는데, 서울에서 11월 황사가 관측된 건 2명0년 이후 처음입니다. 황사는 내일까지 영향을 주면서, 서쪽 지역을 중심으로 일시적으로 농도가 높아질 수 있습니다. 한편, 오늘 서울의 낮 기온은 15도로 평년보다 조금 낮았지만, 햇볕이 강하게 내리쬐면서 저마다 체감하는 날씨는 달랐습니다. \n\n하지만 내일 아침에는 좀 더 옷차림에 신경 쓰셔야 합니다. 북쪽에서 찬 공기가 내려온 데다, 밤사이 구름 없는 하늘로 지면의 열이 빠르게 빠져나가면서 내일 아침 기온은 오늘보다 5도에서 8도가량 뚝 떨어지겠습니다. 그러나 이 추위는 오래가진 않습니다. 고기압이 동쪽으로 물러나면서 수요일 부턴 남풍이 불어 기온이 다시 오르겠고, 목요일에는 기압골이 지나면서 비가 내리겠습니다. 문제는 이 비가 그친 뒤입니다. 북서쪽에서 강한 찬 공기가 밀려와 1.5km 상공의 기온이 하루 만에 10도 이상 곤두박질치겠는데요. 서울도 아침 기온이 토요일 영하 1도까지 떨어져 올가을 처음으로 영하권에 들 가능성이 있습니다. \n\nMBC 뉴스 윤태구입니다.","내일 아침 서울이 영하 1도까지 떨어질 수 있는 것으로 예보되는 가운데, 오늘은 찬 공기와 함께 미세먼지가 문제가 되고 있습니다.","오늘 아침, 도심 하늘이 뿌옇게 흐려졌습니다. 내몽골고원에서 시작된 황사가 바람을 타고 날아들면서 수도권 미세먼지 농도가 ‘매우 나쁨’ 수준까지 치솟았는데요, 11월에 서울에서 황사가 관측된 것은 3년 만에 처음입니다. 내일까지 이어질 황사 영향과 함께, 내일 아침에는 기온도 크게 떨어져 옷차림에 각별한 주의가 필요하겠습니다. 자세한 날씨 소식, 이어서 전해드립니다."
"역대 가장 늦게 찾은 태풍 '참피'‥제주 향해 북상, 최대 60mm 비 더 온다",NaN,"MBC 뉴스의 김대준 기자입니다. 오늘은 태풍 '참피'의 강력한 영향력이 타이완을 강타하고 있습니다. 초속 40미터 이상의 강풍이 도심을 휩쓸며, 건물 주차장이 무너지고 차량이 파손되는 아수라극이 벌어졌습니다. 태풍의 파괴력은 여전히 강력하며, 타이완에서는 이미 1명이 숨고 600여 명이 다쳤습니다. 현재 태풍은 일본 가고시마 부근으로 북상 중이며, 내일 새벽 제주에 가장 가까워질 것으로 예상됩니다. 태풍의 영향으로 제주에서는 300밀리미터 넘는 폭우가 쏟아졌고, 전선이 늘어나고 신호등이 고장나는 등 피해가 속출하고 있습니다. 기상청은 내일까지 비와 강풍이 이어질 것으로 예상하며, 제주 지역에는 최대 60밀리미터의 비가 내릴 것으로 예상하고 있습니다. 태풍의 간접 영향으로 경남 남해안 지역에서는 최대 40밀리미터의 비가 더 오를 것으로 예상됩니다. 기상청은 모레부터는 전국이 대체로 맑아 가을 날씨가 찾아올 것으로 전망하고 있습니다. 태풍 '참피'의 위험한 영향력에 주의를 기울여 주시기 바랍니다. MBC 뉴스 김대준 기자입니다.","태풍 '참피'가 30호 태풍으로 분류되는 순간까지 남아있던 기록을 깨고, 오늘 제주에 가장 가

In [10]:
# ▶ ①-2. 자동 체크 요약 + 웹 비교표 (weather_report.html)
display(summary(weather, ans_w))
make_report(weather, ans_w, 'weather_report.html', '스포츠로 배운 모델 × 날씨 리포트',
            '내 모델은 스포츠 리포트만 학습했어요. 날씨에서도 끝인사 없이 리드 한두 문장으로 시작하는지 보세요.')

,답한 문제,끝인사 없음,2문장 이하,길이 비슷(0.5~2배),본문에 없는 숫자 없음,평균 글자 수
모델,,,,,,
원본 Qwen2.5-3B,5/5,3/5,0/5,0/5,5/5,526
내가 학습한 모델,5/5,5/5,1/5,5/5,5/5,105
gpt-4.1-mini,5/5,5/5,0/5,5/5,4/5,168


✅ 비교표 완성: C:\workspace\ai-datajournalism-labs\4_파인튜닝\weather_report.html


## ② Open-Meteo 날씨 데이터 → 리드
[Open-Meteo](https://open-meteo.com)는 가입 없이 쓰는 무료 날씨 API예요. 내일 예보 **숫자**만 메모로 만들어 넣어요.
정답 리드가 없으니, **메모에 없는 숫자를 지어내지 않았는지**가 가장 중요한 체크예요.

In [4]:
# ▶ ②-1. Open-Meteo에서 내일 예보 받아 '날씨 메모' 만들기
CITIES = {'서울': (37.5665, 126.9780), '강릉': (37.7519, 128.8761), '대전': (36.3504, 127.3845),
          '부산': (35.1796, 129.0756), '제주': (33.4996, 126.5312)}
WMO = {0: '맑음', 1: '대체로 맑음', 2: '구름 조금', 3: '흐림', 45: '안개', 48: '안개', 51: '이슬비', 53: '이슬비', 55: '이슬비',
       61: '비', 63: '비', 65: '강한 비', 71: '눈', 73: '눈', 75: '강한 눈', 80: '소나기', 81: '소나기', 82: '강한 소나기', 95: '뇌우'}

def forecast(lat, lon):
    url = 'https://api.open-meteo.com/v1/forecast'
    p = {'latitude': lat, 'longitude': lon, 'timezone': 'Asia/Seoul', 'forecast_days': 2, 'wind_speed_unit': 'ms',
         'daily': 'weather_code,temperature_2m_max,temperature_2m_min,precipitation_probability_max,precipitation_sum,wind_speed_10m_max'}
    return requests.get(url, params=p, timeout=15).json()['daily']

lines, rows = [], []
for city, (lat, lon) in CITIES.items():
    d = forecast(lat, lon); i = 1                     # 0 = 오늘, 1 = 내일
    t_min, t_max = round(d['temperature_2m_min'][i]), round(d['temperature_2m_max'][i])
    t_min0 = round(d['temperature_2m_min'][0])
    rows.append({'도시': city, '날씨': WMO.get(d['weather_code'][i], '기타'), '최저': t_min, '최고': t_max, '오늘 최저': t_min0,
                 '강수확률(%)': d['precipitation_probability_max'][i], '강수량(mm)': d['precipitation_sum'][i], '최대풍속(m/s)': round(d['wind_speed_10m_max'][i])})
    lines.append(f"{city}: {WMO.get(d['weather_code'][i], '기타')}, 아침 최저 {t_min}도(오늘 {t_min0}도), 낮 최고 {t_max}도, "
                 f"강수확률 {d['precipitation_probability_max'][i]}%, 예상 강수량 {d['precipitation_sum'][i]}mm, 최대 풍속 초속 {round(d['wind_speed_10m_max'][i])}m")
date = d['time'][1]
memo = f"[{date} 내일 날씨 예보 · 출처: Open-Meteo]\n" + '\n'.join(lines)
print(memo)
pd.DataFrame(rows).set_index('도시')

[2026-11-13 내일 날씨 예보 · 출처: Open-Meteo]
서울: 흐림, 아침 최저 8도(오늘 6도), 낮 최고 17도, 강수확률 8%, 예상 강수량 0.0mm, 최대 풍속 초속 1m
강릉: 대체로 맑음, 아침 최저 9도(오늘 5도), 낮 최고 17도, 강수확률 0%, 예상 강수량 0.0mm, 최대 풍속 초속 3m
대전: 흐림, 아침 최저 7도(오늘 2도), 낮 최고 19도, 강수확률 30%, 예상 강수량 0.0mm, 최대 풍속 초속 3m
부산: 구름 조금, 아침 최저 9도(오늘 9도), 낮 최고 18도, 강수확률 4%, 예상 강수량 0.0mm, 최대 풍속 초속 2m
제주: 이슬비, 아침 최저 15도(오늘 11도), 낮 최고 20도, 강수확률 50%, 예상 강수량 0.4mm, 최대 풍속 초속 3m


,날씨,최저,최고,오늘 최저,강수확률(%),강수량(mm),최대풍속(m/s)
도시,,,,,,,
서울,흐림,8,17,6,8,0.0,1
강릉,대체로 맑음,9,17,5,0,0.0,3
대전,흐림,7,19,2,30,0.0,3
부산,구름 조금,9,18,9,4,0.0,2
제주,이슬비,15,20,11,50,0.4,3


In [5]:
# ▶ ②-2. 메모로 세 모델에 리드 쓰게 하기 + 웹 비교표 (meteo_report.html)
meteo = pd.DataFrame([{'title': f'{date} 내일 날씨 (Open-Meteo)', 'input': memo,
                       'instruction': '아래 날씨 예보 메모를 읽고, 앵커가 읽을 내일 날씨 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'}])
ans_m = ask_all(meteo, f'answers_meteo_{date}.json')
for n, a in ans_m.items():
    print(f'■ {n}\n{a[0]}\n')
make_report(meteo, ans_m, 'meteo_report.html', '스포츠로 배운 모델 × 날씨 데이터',
            '정답 리드가 없어요. 메모에 없는 숫자를 지어내지 않았는지, MBC 리드 말투인지 보세요.', gold_col='output', src_label='날씨 메모')

묻는 중: 원본 Qwen2.5-3B
묻는 중: 내가 학습한 모델
묻는 중: gpt-4.1-mini
✅ 저장: answers_meteo_2026-11-13.json
■ 원본 Qwen2.5-3B
안녕하세요, MBC 뉴스의 앵커입니다. 오늘은 날씨에 대한 중요한 정보를 전해드리러 왔습니다. 내일, 서울에서는 흐린 날씨가 예상되며, 아침 최저 기온은 8도로 예상되며, 낮 최고 기온은 17도까지 오를 것으로 보입니다. 강수확률은 8%로 매우 낮으며, 예상 강수량은 0.0mm로, 비가 오지 않을 것으로 예상됩니다. 최대 풍속은 초속 1m까지 오를 수 있습니다.

강릉에서는 대체로 맑은 날씨가 예상되며, 아침 최저 기온은 9도로, 낮 최고 기온은 17도까지 오를 것으로 예상됩니다. 강수확률은 0%로, 비가 오지 않을 것으로 예상됩니다. 예상 강수량은 0.0mm이며, 최대 풍속은 초속 3m까지 오를 수 있습니다.

대전에서는 흐린 날씨가 예상되며, 아침 최저 기온은 7도로, 낮 최고 기온은 19도까지 오를 것으로 예상됩니다. 강수확률은 30%로, 비가 오는 가능성이 있습니다. 예상 강수량은 0.0mm이며, 최대 풍속은 초속 3m까지 오를 수 있습니다.

부산에서는 구름이 조금씩 끼어 흐린 날씨가 예상되며, 아침 최저 기온은 9도로, 낮 최고 기온은 18도까지 오를 것으로 예상됩니다. 강수확률은 4%로, 비가 오는 가능성이 있습니다. 예상 강수량은 0.0mm이며, 최대 풍속은 초속 2m까지 오를 수 있습니다.

제주에서는 이슬비가 예상되며, 아침 최저 기온은 15도로, 낮 최고 기온은 20도까지 오를 것으로 예상됩니다. 강수확률은 50%로, 비가 오는 가능성이 있습니다. 예상 강수량은 0.4mm이며, 최대 풍속은 초속 3m까지 오를 수 있습니다.

이처럼 내일의 날씨는 대부분 흐린 날씨가 예상되며, 비가 오는 지역은 제주와 대전에 해당됩니다. 이러한 정보를 참고하여, 내일의 일정을 계획하시기 바랍니다. 감사합니다.

■ 내가 학습한 모델
내일은 서해를 중심으로 이슬이 내릴 것

## 무엇을 봐야 할까
| 질문 | ① 날씨 리포트 | ② 날씨 데이터 |
|---|---|---|
| 내 모델이 **끝인사 없이** 리드로 시작하나? | 스포츠에서 배운 형식이 날씨에도 나오나 | 데이터만 줘도 나오나 |
| 원본과 **말투**가 다른가? | 원본은 "MBC 뉴스 ○○○입니다"로 시작하나 | |
| **숫자**가 맞나? | 본문과 대조 | 메모의 기온·강수확률과 대조 |
| 스포츠 말투가 **엉뚱하게** 섞이나? | "승리", "연승" 같은 표현 | |

⚠ `answers_*.json`, `*_report.html`, `test_weather.jsonl`에는 MBC 기사 내용이 들어 있어요. GitHub에 올리지 않아요.